In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "CompareSoundings"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "WET"):
        SimulationTime = ("2022-06-30","2022-07-03")
    return SimulationTime

In [ ]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "WET"):
        SimulationTime = ("2022-06-30","2022-07-03")
    elif (RunType[0] == "TRACER") and (RunType[1] == "DRY"):
        SimulationTime = ("2022-06-08","2022-06-11")
    return SimulationTime

spinup_hours = 24
# spinup_hours = 12

# RunType = ("TRACER","WET","NSSL",spinup_hours)
# RunType = ("TRACER","WET","TEMPO",spinup_hours)
# RunType = ("TRACER","DRY","NSSL",spinup_hours)
RunType = ("TRACER","DRY","TEMPO",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

In [ ]:
####################################
#READING REAL SOUNDING TIMESTAMPS

In [ ]:
import glob
import os
import re
from datetime import datetime

FolderPath = "/glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA/Observation_Data/TRACER/Soundings/housondewnpnM1.b1/"

# Get all .cdf files
file_list = glob.glob(FolderPath + "*.cdf")

# Extract datetime strings from filenames
times = []
for f in file_list:
    # Extract the part like 20220609.233000
    match = re.search(r'\.(\d{8}\.\d{6})\.cdf$', os.path.basename(f))
    if match:
        dt_str = match.group(1)
        # Convert to Python datetime object
        dt = datetime.strptime(dt_str, "%Y%m%d.%H%M%S")
        times.append(dt)

# Sort chronologically
times.sort()

In [ ]:
####################################
#READING MODEL SOUNDING TIMESTAMPS

In [ ]:
from datetime import datetime
time_strings = [t.replace(":", ".") for t in ModelData.timeStrings]
model_times = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]

In [ ]:
####################################
#FINDING TIME MATCHES

In [ ]:
from datetime import timedelta

tolerance = timedelta(minutes=10)
matches = []

for i, s in enumerate(times):
    diffs = [abs(m - s) for m in model_times]
    min_idx = diffs.index(min(diffs))
    closest = model_times[min_idx]

    if abs(closest - s) <= tolerance:
        delta = abs(closest - s)
        matches.append((i, min_idx, s, closest, delta))

# Print matches
for sound_idx, model_idx, s, m, delta in matches:
    print(f"[sound {sound_idx:02d}] {s} ↔ [model {model_idx:03d}] {m} (Δ={delta})")

# Output matched indices
sound_inds = [s_idx for (s_idx, _, _, _, _) in matches]
model_inds = [m_idx for (_, m_idx, _, _, _) in matches]


In [ ]:
####################################
#GETTING REAL SOUNDING DATA

In [ ]:
sound_0 = xr.open_dataset(file_list[0])
lat_sound, lon_sound = sound_0['lat'].isel(time=0).data, sound_0['lon'].isel(time=0).data
lat_sound, lon_sound

In [ ]:
def GetRealSoundingData(file_list, i):
    ds = xr.open_dataset(file_list[i])
    T_sound = ds['tdry'].values     # °C
    P_sound = ds['pres'].values     # hPa
    z_sound = ds['alt'].values      # m

    return T_sound,P_sound,z_sound

In [ ]:
####################################
#GETTING MODEL SOUNDING DATA

In [ ]:
data_0 = ModelData.GetDataTimestep(t=0)
lat_model = data_0['latitude'].data
lon_model = data_0['longitude'].data

In [ ]:
def theta_to_temperature(theta, pressure, p0=1000.0):
    """
    Convert potential temperature (θ) to actual temperature (T) given pressure.
    """
    Rd = 287.0   # J/(kg*K)
    Cp = 1004.0  # J/(kg*K)
    
    theta = np.asarray(theta)
    pressure = np.asarray(pressure)
    
    T = theta * (pressure / p0) ** (Rd / Cp)
    return T


In [ ]:
# def GetModelSoundingData(model_ind,i,j):
#     data = ModelData.GetDataTimestep(t=model_ind)
    
#     TH_model = data['theta'].data[:,i,j]
#     P_model = data['pressure'].data[:,i,j]/100
#     T_model = theta_to_temperature(TH_model, P_model)
#     return T_model,P_model

def GetModelSoundingData(model_ind,lat_sound,lon_sound):
    data = ModelData.GetDataTimestep(t=model_ind)
    
    # Use xarray's .sel with "method='nearest'"
    point = data.sel(latitude=lat_sound, longitude=lon_sound, method='nearest')
    
    # Then extract variables
    TH_model = point['theta'].data
    P_model  = point['pressure'].data / 100  # convert Pa → hPa
    T_model  = theta_to_temperature(TH_model, P_model) - 273.15
    
    return T_model, P_model

In [ ]:
####################################
#getting the model index for lat/lon of real sounding 

In [ ]:
# def find_nearest_latlon(lat_model, lon_model, lat_sound, lon_sound):
#     """
#     Find the nearest grid index to a target latitude and longitude.
#     Works for 1D latitude and longitude arrays (structured grid).
#     """
#     i = np.abs(lat_model - lat_sound).argmin()  # nearest latitude index
#     j = np.abs(lon_model - lon_sound).argmin()  # nearest longitude index
#     return i, j

# data0 = ModelData.GetDataTimestep(t=0)
# lat_model = data0['latitude'].data
# lon_model = data0['longitude'].data

# i,j = find_nearest_latlon(lat_model, lon_model, lat_sound, lon_sound)
# print(data0['latitude'][i].data,data0['longitude'][j].data)

In [ ]:
####################################
#PLOTTING COMPARISONS

In [ ]:
#################
#naive calculation: looking at single timestep, not following the baloon through top
#################

In [ ]:
def PlotAllSoundingComparisons(sound_inds, model_inds, file_list, lat_sound, lon_sound,
                               GetRealSoundingData, GetModelSoundingData, times,
                               ncols=3, figsize=(13, 11)):
    """
    Create subplots comparing observed vs. model temperature profiles for any number of soundings.

    Parameters
    ----------
    sound_inds, model_inds : list[int]
        Lists of indices for soundings and corresponding model timesteps.
    file_list : list[str]
        List of sounding file paths.
    lat_sound, lon_sound : float
        Latitude and longitude of the sounding site (for model extraction).
    GetRealSoundingData : function
        Function returning (T_sound [°C], P_sound [hPa]).
    GetModelSoundingData : function
        Function returning (T_model [°C], P_model [hPa]).
    times : list of datetime
        Sounding launch times.
    ncols : int, optional
        Number of subplot columns (default 3).
    figsize : tuple, optional
        Figure size.
    """

    nplots = len(sound_inds)
    nrows = int(np.ceil(nplots / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=True, sharey=True)
    axes = np.atleast_1d(axes).flatten()

    for k, (sound_ind, model_ind) in enumerate(zip(sound_inds, model_inds)):
        ax = axes[k]

        # --- Load data ---
        T_sound, P_sound, z_sound = GetRealSoundingData(file_list, sound_ind)
        T_model, P_model = GetModelSoundingData(model_ind, lat_sound, lon_sound)

        # --- Plot ---
        ax.plot(T_sound, P_sound, color="blue", label="Sounding")
        ax.plot(T_model, P_model, color="red", linestyle="solid", label="Model")
        ax.invert_yaxis()
        # ax.plot(T_sound, z_sound, color="blue", label="Sounding")
        # ax.plot(T_model, z_model, color="red", linestyle="--", label="Model")

        # --- Titles and labels ---
        title_time = times[sound_ind].strftime('%Y-%m-%d %H:%M')
        ax.set_title(title_time, fontsize=9)
        ax.grid(True)

        if k % ncols == 0:
            ax.set_ylabel("Pressure (hPa)")
            # ax.set_ylabel("z (m)")
        if k >= (nrows - 1) * ncols:
            ax.set_xlabel("Temperature (°C)")

    # Hide any unused axes
    for ax in axes[nplots:]:
        ax.axis("off")

    # Global legend and layout
    fig.legend(["Sounding", "Model"], loc="upper right", fontsize=10)
    fig.suptitle("Model vs Observed Temperature Soundings", fontsize=14)
    plt.tight_layout(rect=[0, 0, 0.95, 0.96])
    plt.show()


In [ ]:
PlotAllSoundingComparisons(
    sound_inds, model_inds,
    file_list, lat_sound, lon_sound,
    GetRealSoundingData, GetModelSoundingData, times,
    ncols=4  # automatically adjusts rows
)

In [ ]:
#################
#full calculation: following the timesteps of the balloon
#################

In [ ]:
zgrid = ModelData.initData['zgrid']
z_model= zgrid.sel(latitude=lat_sound, longitude=lon_sound, method='nearest').data

In [ ]:
def GetSounding(i):
    sound = xr.open_dataset(file_list[i])
    T_sound,P_sound,z_sound = GetRealSoundingData(file_list, i)
    
    sound_times = sound['time'].values   # e.g., np.array([...], dtype='datetime64[ns]')
    import numpy as np
    from datetime import datetime, timedelta
    
    # Convert numpy.datetime64 → Python datetime
    sound_times_dt = [datetime.utcfromtimestamp(t.astype('datetime64[s]').astype(int))
                      for t in sound_times]

    ############################################
    closest_model_times = []
    closest_model_indices = []
    
    for s in sound_times_dt:
        diffs = [abs(m - s) for m in model_times]
        idx = diffs.index(min(diffs))     # index of nearest model time
        closest_model_indices.append(idx)
        closest_model_times.append(model_times[idx])
    closest_model_indices = [model_times.index(t) for t in closest_model_times]
    
    # Sounding and model heights (in meters)
    z_sound = sound['alt'].data            # shape: (n_sounding_levels,)                 # 1D array of model heights (e.g. shape (nz,))
    
    # Find closest model index for each sounding height
    z_indices = [np.abs(z_model - z).argmin() for z in z_sound]
    z_indices = np.clip(z_indices, 0, len(z_model) - 2)

    return closest_model_indices,z_indices, T_sound,P_sound

In [ ]:
def GetModelSoundingData_V2(closest_model_indices,z_indices):

    last_index = None
    for model_ind in closest_model_indices:
        if model_ind != last_index:
            point = ModelData.GetDataTimestep(t=model_ind).sel(latitude=lat_sound, longitude=lon_sound, method='nearest')
            point = point.isel(nVertLevels=z_indices)
            last_index = model_ind
    
    # Use xarray's .sel with "method='nearest'"
    
    # Then extract variables
    TH_model = point['theta'].data
    P_model  = point['pressure'].data / 100  # convert Pa → hPa
    T_model  = theta_to_temperature(TH_model, P_model) - 273.15
    return T_model, P_model

In [ ]:
def SaveFigure(fig, plottype, dpi=150, extension="png"):
    """
    Saves a matplotlib Figure to a subdirectory named after the model configuration.
    """

    # Ensure dpi is a plain Python int
    dpi = int(np.atleast_1d(dpi)[0])  # Handles np.float64 or array inputs safely

    # --- Define output subdirectory ---
    outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{ModelData.mpType}_{ModelData.spinup_hours}hrs"
    save_dir = os.path.join(outputPlottingDirectory, outputSubDirectory)
    os.makedirs(save_dir, exist_ok=True)

    # --- File path ---
    outputFile = os.path.join(
        save_dir,
        f"CompareSoundings_{plottype}.{extension}"
    )

    # --- Save and close ---
    fig.savefig(outputFile, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure to: {outputFile}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

nplots = len(sound_inds)
ncols = 4
nrows = int(np.ceil(nplots / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 5 * nrows), sharex=True, sharey=True)
axes = axes.flatten()

for k, sound_ind in enumerate(sound_inds):
    ax = axes[k]
    closest_model_indices, z_indices, T_sound, P_sound = GetSounding(i=sound_ind)
    T, P = GetModelSoundingData_V2(closest_model_indices, z_indices)

    ax.plot(T, P, color='red', label='Model')
    ax.plot(T_sound, P_sound, color='blue', label='Sounding')
    ax.invert_yaxis()
    # --- Titles and labels ---
    title_time = times[sound_ind].strftime('%Y-%m-%d %H:%M')
    ax.set_title(title_time, fontsize=9)
    ax.grid(True)
    if k % ncols == 0:
        ax.set_ylabel("Pressure (hPa)")
    if k >= (nrows - 1) * ncols:
        ax.set_xlabel("Temperature (°C)")
    # break

# Hide any unused axes
for ax in axes[nplots:]:
    ax.axis('off')

fig.legend(['Model', 'Sounding'], loc='upper right')
plt.suptitle("Model vs Sounding Temperature Profiles", fontsize=16)
plt.tight_layout(rect=[0, 0, 0.95, 0.96])

SaveFigure(fig, plottype="temperature")

In [ ]:
#same process but for zonal wind

In [ ]:
def GetRealSoundingData(file_list, i):
    ds = xr.open_dataset(file_list[i])
    # T_sound = ds['tdry'].values     # °C
    U_sound = ds['u_wind'].values     # m/s
    V_sound = ds['v_wind'].values     # m/s
    P_sound = ds['pres'].values     # hPa
    z_sound = ds['alt'].values      # m

    return U_sound,V_sound,P_sound,z_sound

def GetSounding(i):
    sound = xr.open_dataset(file_list[i])
    U_sound,V_sound,P_sound,z_sound = GetRealSoundingData(file_list, i)
    
    sound_times = sound['time'].values   # e.g., np.array([...], dtype='datetime64[ns]')
    import numpy as np
    from datetime import datetime, timedelta
    
    # Convert numpy.datetime64 → Python datetime
    sound_times_dt = [datetime.utcfromtimestamp(t.astype('datetime64[s]').astype(int))
                      for t in sound_times]

    ############################################
    closest_model_times = []
    closest_model_indices = []
    
    for s in sound_times_dt:
        diffs = [abs(m - s) for m in model_times]
        idx = diffs.index(min(diffs))     # index of nearest model time
        closest_model_indices.append(idx)
        closest_model_times.append(model_times[idx])
    closest_model_indices = [model_times.index(t) for t in closest_model_times]
    
    # Sounding and model heights (in meters)
    z_sound = sound['alt'].data            # shape: (n_sounding_levels,)                 # 1D array of model heights (e.g. shape (nz,))
    
    # Find closest model index for each sounding height
    z_indices = [np.abs(z_model - z).argmin() for z in z_sound]
    z_indices = np.clip(z_indices, 0, len(z_model) - 2)

    return closest_model_indices,z_indices, U_sound,V_sound,P_sound

def GetModelSoundingData_V2(closest_model_indices,z_indicies):

    last_index = None
    for model_ind in closest_model_indices:
        if model_ind != last_index:
            point = ModelData.GetDataTimestep(t=model_ind).sel(latitude=lat_sound, longitude=lon_sound, method='nearest')
            point = point.isel(nVertLevels=z_indicies)
            last_index = model_ind
    
    # Use xarray's .sel with "method='nearest'"
    
    # Then extract variables
    # TH_model = point['theta'].data
    P_model  = point['pressure'].data / 100  # convert Pa → hPa
    # T_model  = theta_to_temperature(TH_model, P_model) - 273.15
    U_model = point['uReconstructZonal'].data
    V_model = point['uReconstructMeridional'].data
    return U_model,V_model,P_model

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

nplots = len(sound_inds)
ncols = 4
nrows = int(np.ceil(nplots / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 5 * nrows), sharex=True, sharey=True)
axes = axes.flatten()

for k, sound_ind in enumerate(sound_inds):
    ax = axes[k]
    closest_model_indices, z_indices, U_sound, V_sound, P_sound = GetSounding(i=sound_ind)
    U, V, P = GetModelSoundingData_V2(closest_model_indices, z_indices)

    ax.plot(U, P, color='red', label='Model')
    ax.plot(U_sound, P_sound, color='blue', label='Sounding')
    ax.invert_yaxis()
    # --- Titles and labels ---
    title_time = times[sound_ind].strftime('%Y-%m-%d %H:%M')
    ax.set_title(title_time, fontsize=9)
    ax.grid(True)
    if k % ncols == 0:
        ax.set_ylabel("U (m/s)")
    if k >= (nrows - 1) * ncols:
        ax.set_xlabel("Temperature (°C)")
    # break

# Hide any unused axes
for ax in axes[nplots:]:
    ax.axis('off')

fig.legend(['Model', 'Sounding'], loc='upper right')
plt.suptitle("Model vs Sounding Temperature Profiles", fontsize=16)
plt.tight_layout(rect=[0, 0, 0.95, 0.96])

SaveFigure(fig, plottype="uwind")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

nplots = len(sound_inds)
ncols = 4
nrows = int(np.ceil(nplots / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 5 * nrows), sharex=True, sharey=True)
axes = axes.flatten()

for k, sound_ind in enumerate(sound_inds):
    ax = axes[k]
    closest_model_indices, z_indices, U_sound, V_sound, P_sound = GetSounding(i=sound_ind)
    U, V, P = GetModelSoundingData_V2(closest_model_indices, z_indices)

    ax.plot(V, P, color='red', label='Model')
    ax.plot(V_sound, P_sound, color='blue', label='Sounding')
    ax.invert_yaxis()
    # --- Titles and labels ---
    title_time = times[sound_ind].strftime('%Y-%m-%d %H:%M')
    ax.set_title(title_time, fontsize=9)
    ax.grid(True)
    if k % ncols == 0:
        ax.set_ylabel("V (m/s)")
    if k >= (nrows - 1) * ncols:
        ax.set_xlabel("Temperature (°C)")
    # break

# Hide any unused axes
for ax in axes[nplots:]:
    ax.axis('off')

fig.legend(['Model', 'Sounding'], loc='upper right')
plt.suptitle("Model vs Sounding Temperature Profiles", fontsize=16)
plt.tight_layout(rect=[0, 0, 0.95, 0.96])

SaveFigure(fig, plottype="vwind")